# GSK_HepG2 ablation: metric vs deltaprop candidate-pool size

Pulls the single run from the `ablation_gsk_hepg2_candidates` wandb project, recovers the per-(candidate_size, model) rows from its history, and plots deltaprop across candidate sizes with the chemprop baseline as a horizontal reference line.

Each `wandb.log(row)` call in the ablation became a separate history step, so we read the full history with `scan_history()` and split on the `model` column ourselves. chemprop is invariant to `candidate_size`, so it appears once with `candidate_size = NaN`.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

PROJECT = "ablations"
RUN_PREFIX = "db_malaria_candidate_size"
RUN_NAME = None  # set to an exact run name to pin a specific run; otherwise use the latest
METRIC = "test_average_precision"  # swap for test_mcc, test_f1, test_roc_auc, ...

api = wandb.Api()
runs = api.runs(f"{api.default_entity}/{PROJECT}")
if RUN_NAME is not None:
    run = next(r for r in runs if r.name == RUN_NAME)
else:
    # Runs are named '<prefix>_<timestamp>'; pick the most recent matching run.
    matches = [r for r in runs if r.name.startswith(RUN_PREFIX)]
    run = max(matches, key=lambda r: r.created_at)
run.name, run.id

In [ ]:
# Full, unsampled history -> one row per (candidate_size, model).
df = pd.DataFrame(run.scan_history())
df = df[["candidate_size", "model", "n_train", METRIC]].sort_values(["model", "candidate_size"])
df

In [ ]:
delta = df[df["model"] == "deltaprop"].sort_values("candidate_size")
chemprop = df[df["model"] == "chemprop"]

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(delta["candidate_size"], delta[METRIC], marker="o", label="deltaprop")

# if len(chemprop):
#     ax.axhline(
#         chemprop[METRIC].iloc[0],
#         color="gray",
#         linestyle="--",
#         label="chemprop (baseline)",
#     )

# ax.set_xscale("log", base=2)
ax.set_xticks(delta["candidate_size"])
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("candidate size")
ax.set_ylabel(METRIC)
ax.set_title("DB_MALARIA candidate size ablation (SCAFFOLD split)")
ax.grid(True)
ax.legend()
fig.tight_layout()
# fig.savefig("gsk_hepg2_candidates_ablation.png", dpi=150)
plt.show()